# Assignment 1: run the experiments (E1–E8) on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antonistsiorvas/cifar10-classifiers/blob/main/assignment-1/colab_experiments.ipynb)

**How to use**
1. Open this notebook from GitHub in Colab (the badge above, or *File → Open notebook → GitHub*).
2. *Runtime → Change runtime type → **T4 GPU***.
3. Run the cells from top to bottom. Every finished run is saved on Google Drive right away.

**After a disconnect:** run cells 1–6 again, then continue with the next experiment cell.
Runs that already finished are skipped, so nothing is lost except the run that was interrupted.

## 1. Check the GPU
Prints the PyTorch version and the GPU. Without a GPU everything still works, but very slowly.

In [ ]:
import torch

print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NO GPU! Select one: Runtime → Change runtime type → T4 GPU, then run this cell again.")

## 2. Mount Google Drive
The results are stored in `MyDrive/cifar10-classifiers/assignment-1-results/`, which survives
disconnects and new sessions.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_RESULTS = Path("/content/drive/MyDrive/cifar10-classifiers/assignment-1-results")
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print("Results folder on Drive:", DRIVE_RESULTS)

## 3. Get the code
Clones the repository into `/content/cifar10-classifiers`, or updates it with `git pull` if it
is already there.

In [ ]:
import os

REPO = Path("/content/cifar10-classifiers")
if (REPO / ".git").exists():
    !git -C /content/cifar10-classifiers pull
else:
    !git clone https://github.com/antonistsiorvas/cifar10-classifiers /content/cifar10-classifiers
%cd /content/cifar10-classifiers
!git log -1 --oneline

## 4. Install missing packages
Colab already has PyTorch with CUDA, NumPy and pandas, so normally nothing is installed.
PyTorch is never reinstalled (that would replace Colab's CUDA build).

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is missing; this should not happen on Colab")
needed = {"numpy": "numpy", "pandas": "pandas"}  # import name -> pip name
missing = [pip for module, pip in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Installed:", ", ".join(missing))
else:
    print("Nothing to install")

## 5. Download CIFAR-10
Downloads and extracts the dataset into the repo's `data/` folder (skipped if it is already there).

In [ ]:
sys.path.insert(0, str(REPO))
from common.cifar10 import DATA_DIR, download_cifar10

download_cifar10()
print(sorted(p.name for p in (DATA_DIR / "cifar-10-batches-py").iterdir()))

## 6. Save the results on Drive
Replaces `assignment-1/results/` in the repo with a link to the Drive folder, so every run is
written to Drive the moment it finishes. If the repo already contains committed results, the
ones Drive does not have yet are copied there first.

In [ ]:
import shutil

link = REPO / "assignment-1" / "results"
if link.is_symlink():
    link.unlink()
elif link.exists():
    for src in link.rglob("*"):
        dst = DRIVE_RESULTS / src.relative_to(link)
        if src.is_file() and not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
    shutil.rmtree(link)
link.symlink_to(DRIVE_RESULTS, target_is_directory=True)

done = len(list(DRIVE_RESULTS.glob("runs/*/metrics.json")))
print(f"{link} -> {os.readlink(link)}")
print(f"Finished runs already on Drive: {done}")

## 7. Quick test and time estimate
Runs E1 and E5 in smoke mode (2,000 training images, 2 epochs, written to `results_smoke/`, not
to Drive) to check that everything works, then estimates the epoch times on the full 45,000
images and the total time for all runs.

In [ ]:
import csv
import statistics

SMOKE = REPO / "assignment-1" / "results_smoke"
shutil.rmtree(SMOKE, ignore_errors=True)  # always measure on this machine
!cd /content/cifar10-classifiers && python -u assignment-1/experiments.py --exp E1 E5 --smoke


def full_epoch_time(run_name):
    # Time of the 2nd smoke epoch (the 1st includes warm-up), scaled to 45,000 images
    with open(SMOKE / "runs" / run_name / "history.csv") as f:
        rows = list(csv.DictReader(f))
    return float(rows[-1]["epoch_time_s"]) * 45000 / 2000


sys.path.insert(0, str(REPO / "assignment-1"))
from experiments import DEFAULTS, EXPERIMENTS

all_runs = [{**DEFAULTS, **entry} for runs in EXPERIMENTS.values() for entry in runs]
n_mlp = sum(1 for r in all_runs if r["model"] == "mlp")
n_cnn = len(all_runs) - n_mlp
mlp_epoch = statistics.mean(full_epoch_time(e["name"]) for e in EXPERIMENTS["E1"])
cnn_epoch = full_epoch_time("E5_cnn_ref")
aug_epoch = full_epoch_time("E5_cnn_augment")

print()
print(f"One epoch on 45,000 images (rough): MLP ~{mlp_epoch:.1f} s, CNN ~{cnn_epoch:.1f} s, "
      f"CNN with augmentation ~{aug_epoch:.1f} s")
for epochs in (25, 50):
    total = (n_mlp * mlp_epoch + n_cnn * cnn_epoch) * epochs
    print(f"All {len(all_runs)} runs ({n_mlp} MLP, {n_cnn} CNN) at {epochs} epochs each: "
          f"~{total / 3600:.1f} h")
print("Early stopping usually ends a run well before 50 epochs; batch size 32 and "
      "augmentation runs are slower than average.")

## 8. Run the experiments
One cell per experiment, cheapest first. Each cell can be run on its own (after cells 1–6), in
any session; finished runs are skipped. The output shows every epoch live.

### 8.1 E1: MLP width (1 hidden layer)

In [ ]:
!cd /content/cifar10-classifiers && python -u assignment-1/experiments.py --exp E1

### 8.2 E2: MLP depth × width

In [ ]:
!cd /content/cifar10-classifiers && python -u assignment-1/experiments.py --exp E2

### 8.3 E4: input normalization (raw vs unit)

In [ ]:
!cd /content/cifar10-classifiers && python -u assignment-1/experiments.py --exp E4

### 8.4 E3: activation functions (MLP and CNN)

In [ ]:
!cd /content/cifar10-classifiers && python -u assignment-1/experiments.py --exp E3

### 8.5 E6: learning rate, batch size, optimizer (MLP and CNN)

In [ ]:
!cd /content/cifar10-classifiers && python -u assignment-1/experiments.py --exp E6

### 8.6 E5: CNN variations

In [ ]:
!cd /content/cifar10-classifiers && python -u assignment-1/experiments.py --exp E5

### 8.7 E7: MLP decreasing vs increasing width

In [ ]:
!cd /content/cifar10-classifiers && python -u assignment-1/experiments.py --exp E7

### 8.8 E8: combination of the best choices (MLP and CNN)

In [ ]:
!cd /content/cifar10-classifiers && python -u assignment-1/experiments.py --exp E8

## 9. Results so far
The table from `results/summary.csv` and the number of finished runs.

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "assignment-1"))
from experiments import EXPERIMENTS

results = REPO / "assignment-1" / "results"
finished = {p.parent.name for p in results.glob("runs/*/metrics.json")}
n_total = sum(len(runs) for runs in EXPERIMENTS.values())
print(f"Finished runs: {len(finished)} / {n_total}")
for exp, runs in EXPERIMENTS.items():
    todo = [e["name"] for e in runs if e["name"] not in finished]
    if todo:
        print(f"  {exp}: {len(todo)} left ({', '.join(todo)})")

summary_path = results / "summary.csv"
if summary_path.exists():
    df = pd.read_csv(summary_path).drop_duplicates("name", keep="last")
    columns = ["experiment", "name", "model", "train_acc", "val_acc", "test_acc",
               "best_epoch", "epochs_run", "train_time_s", "n_params"]
    pd.set_option("display.max_rows", 100)
    display(df[columns].round(4))
else:
    print("No summary.csv yet")

## 10. Download the results
Zips the Drive results folder into `MyDrive/cifar10-classifiers/results.zip` and starts the
download. Locally, extract it into `assignment-1/results/` (so that it contains `runs/` and
`summary.csv`) and commit.

In [ ]:
zip_path = shutil.make_archive(str(DRIVE_RESULTS.parent / "results"), "zip", root_dir=DRIVE_RESULTS)
print(f"Created {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

from google.colab import files
files.download(zip_path)